In [95]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, hour, lower,concat,lit, rank, when,unix_timestamp
from pyspark.sql.window import Window

In [12]:
spark = SparkSession.builder \
    .appName("PostgreSQL to PySpark") \
    .config("spark.jars", "C:\\Users\\egor2\\Spark\\jdbc\\postgresql-42.7.7.jar") \
    .getOrCreate()
print(spark)

db_properties = {
    "user": "postgres",
    "password": "1",
    "driver": "org.postgresql.Driver"
}
db_url = "jdbc:postgresql://localhost:5432/films"

In [14]:
def load_table(table_name):
    return spark.read.jdbc(url=db_url, table=table_name, properties=db_properties)

In [16]:
film = load_table("public.film")
film_category = load_table("public.film_category")
category = load_table("public.category")
actor = load_table("public.actor")
film_actor = load_table("public.film_actor")
inventory = load_table("public.inventory")
rental = load_table("public.rental")
payment = load_table("public.payment")
customer = load_table("public.customer")
address = load_table("public.address")
city = load_table("public.city")

In [20]:
film_count_by_category = film_category.join(category, category.category_id == film_category.category_id) \
    .groupBy(category.name.alias("ctaegory_name")) \
    .agg(count("*").alias("count_films")) \
    .orderBy(col("count_films").desc())
film_count_by_category.show()

+-------------+-----------+
|ctaegory_name|count_films|
+-------------+-----------+
|       Sports|         74|
|      Foreign|         73|
|       Family|         69|
|  Documentary|         68|
|    Animation|         66|
|       Action|         64|
|          New|         63|
|        Drama|         62|
|       Sci-Fi|         61|
|        Games|         61|
|     Children|         60|
|       Comedy|         58|
|       Travel|         57|
|     Classics|         57|
|       Horror|         56|
|        Music|         51|
+-------------+-----------+



In [45]:
popular_actors = actor.join(film_actor, film_actor.actor_id == actor.actor_id) \
    .join(inventory, inventory.film_id == film_actor.film_id) \
    .join(rental, rental.inventory_id == inventory.inventory_id) \
    .groupBy(actor.first_name, actor.last_name) \
    .agg(count("*").alias("rent_count")) \
    .select(concat(col("first_name"), lit(" "), col("last_name")).alias("actor_name"), col("rent_count")) \
    .orderBy(col("rent_count").desc()) \
    .limit(10)

popular_actors.show()

+------------------+----------+
|        actor_name|rent_count|
+------------------+----------+
|       SUSAN DAVIS|       825|
|    GINA DEGENERES|       753|
|    MATTHEW CARREY|       678|
|       MARY KEITEL|       674|
|ANGELA WITHERSPOON|       654|
|       WALTER TORN|       640|
|       HENRY BERRY|       612|
|       JAYNE NOLTE|       611|
|        VAL BOLGER|       605|
|     SANDRA KILMER|       604|
+------------------+----------+



In [53]:
#Вывести категорию фильмов, на которую потратили больше всего денег.

most_spend_movies = payment.join(rental, rental.rental_id== payment.rental_id).\
    join(inventory, inventory.inventory_id== rental.inventory_id).\
    join(film, film.film_id == inventory.film_id).\
    join(film_category, film_category.film_id== film.film_id).\
    join(category, category.category_id== film_category.category_id).\
    groupBy(category.name.alias("category_name")).\
    agg(sum(payment.amount).alias("total_spent")).\
    orderBy(col("total_spent").desc())

most_spend_movies.show()

+-------------+-----------+
|category_name|total_spent|
+-------------+-----------+
|       Sports|    5314.21|
|       Sci-Fi|    4756.98|
|    Animation|    4656.30|
|        Drama|    4587.39|
|       Comedy|    4383.58|
|       Action|    4375.85|
|          New|    4361.57|
|        Games|    4281.33|
|      Foreign|    4270.67|
|       Family|    4226.07|
|  Documentary|    4217.52|
|       Horror|    3722.54|
|     Children|    3655.55|
|     Classics|    3639.59|
|       Travel|    3549.64|
|        Music|    3417.72|
+-------------+-----------+



In [67]:
#Вывести названия фильмов, которых нет в inventory. Написать запрос без использования оператора IN.

noninventory_movies = film.join(inventory, film.film_id == inventory.film_id, "left") \
    .where(inventory.film_id.isNull()) \
    .select(film.title)

noninventory_movies.show(truncate=False)

+---------------------+
|title                |
+---------------------+
|CHOCOLATE DUCK       |
|BUTCH PANTHER        |
|VOLUME HOUSE         |
|ORDER BETRAYED       |
|TADPOLE PARK         |
|KILL BROTHERHOOD     |
|FRANKENSTEIN STRANGER|
|CROSSING DIVORCE     |
|SUICIDES SILENCE     |
|CATCH AMISTAD        |
|PERDITION FARGO      |
|FLOATS GARDEN        |
|GUMP DATE            |
|WALLS ARTIST         |
|GLADIATOR WESTWARD   |
|HOCUS FRIDA          |
|ARSENIC INDEPENDENCE |
|MUPPET MILE          |
|FIREHOUSE VIETNAM    |
|ROOF CHAMPION        |
+---------------------+
only showing top 20 rows



In [79]:
act_cnts = category.join(film_category, category.category_id == film_category.category_id) \
    .join(film, film.film_id == film_category.film_id) \
    .join(film_actor, film_actor.film_id == film.film_id) \
    .join(actor, actor.actor_id == film_actor.actor_id) \
    .where(category.name == "Children") \
    .groupBy(actor.first_name, actor.last_name) \
    .agg(count("*").alias("appearances")) \
    .select(concat(col("first_name"), lit(" "), col("last_name")).alias("actor_name"), col("appearances"))

window_spec = Window.orderBy(col("appearances").desc())
rank_act = act_cnts.withColumn("ranking", rank().over(window_spec))

top_actors = rank_act.where(col("ranking") <= 3) \
    .select("actor_name", "appearances")

top_actors.show()

+-------------+-----------+
|   actor_name|appearances|
+-------------+-----------+
| HELEN VOIGHT|          7|
|  SUSAN DAVIS|          6|
|KEVIN GARLAND|          5|
|   RALPH CRUZ|          5|
|  WHOOPI HURT|          5|
|   MARY TANDY|          5|
+-------------+-----------+



In [85]:
act_city = city.join(address, city.city_id == address.city_id) \
    .join(customer, customer.address_id == address.address_id) \
    .groupBy(city.city) \
    .agg(
        count(when(customer.active == 1, True)).alias("active"),
        count(when(customer.active == 0, True)).alias("nonactive")
    ) \
    .orderBy(col("nonactive").desc()) \
    .select("city")

act_city.show()


+----------------+
|            city|
+----------------+
|          Ktahya|
|Charlotte Amalie|
|         Wroclaw|
|       Najafabad|
|     Szkesfehrvr|
|          Kamyin|
|   Coatzacoalcos|
|         Bat Yam|
| Southend-on-Sea|
|        Uluberia|
|       Pingxiang|
|        Xiangfan|
|      Kumbakonam|
|          Daxian|
|          Amroha|
|          Guaruj|
|      Manchester|
|          Laredo|
|Shubra al-Khayma|
|        Bradford|
+----------------+
only showing top 20 rows



In [97]:
rntl_hrs = category.join(film_category, category.category_id == film_category.category_id) \
    .join(film, film.film_id == film_category.film_id) \
    .join(inventory, inventory.film_id == film.film_id) \
    .join(rental, rental.inventory_id == inventory.inventory_id) \
    .join(customer, rental.customer_id == customer.customer_id) \
    .join(address, customer.address_id == address.address_id) \
    .join(city, address.city_id == city.city_id) \
    .groupBy(category.category_id, category.name, city.city) \
    .agg(sum((unix_timestamp(rental.return_date) - unix_timestamp(rental.rental_date)) / 3600).alias("tot_rntl_hrs")) \
    .select(category.name.alias("category_name"), col("tot_rntl_hrs"), city.city)

city_with_a = rntl_hrs.where(col("city").like("A%")) \
    .orderBy(col("tot_rntl_hrs").desc()) \
    .limit(1)

city_double = rntl_hrs.where(col("city").like("%-%")) \
    .orderBy(col("tot_rntl_hrs").desc()) \
    .limit(1)

result = city_with_a.union(city_double)

result.show()

+-------------+------------------+----------+
|category_name|      tot_rntl_hrs|      city|
+-------------+------------------+----------+
|       Action|1109.8666666666666|Avellaneda|
|       Comedy|            898.75| Lapu-Lapu|
+-------------+------------------+----------+

